# 04 — Exploración de Datos (EDA) y Limpieza Dinámica

Este notebook implementa un enfoque profesional de Minería de Datos en dos fases:
1. **Fase 1: Análisis Exploratorio de Datos (EDA) General**. Se perfila la base de datos para identificar el porcentaje de nulos, inconsistencias lógicas y se calculan estadísticos robustos (cuartiles, IQR) para la detección de outliers.
2. **Fase 2: Limpieza Dinámica**. Con base en los resultados empíricos reportados en la fase 1, el código determina la estrategia (imputar vs eliminar) y ejecuta la limpieza estructurada.

In [10]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from py4j.java_gateway import java_import

print("=" * 60)
print("CELDA 1: Inicialización Spark + Configuración Snowflake")
print("=" * 60)

spark = SparkSession.builder.appName("NYC_Taxi_EDA_Cleaning").getOrCreate()
print("✓ SparkSession creada")

SF_OPTIONS = {
    "sfURL":       f"{os.environ['SF_ACCOUNT']}.snowflakecomputing.com",
    "sfUser":      os.environ["SF_USER"],
    "sfPassword":  os.environ["SF_PASSWORD"],
    "sfDatabase":  os.environ["SF_DATABASE"],
    "sfSchema":    os.environ["SF_SCHEMA_RAW"],
    "sfWarehouse": os.environ["SF_WAREHOUSE"],
    "sfRole":      os.environ["SF_ROLE"],
}
SF_OPTS_ANALYTICS = {**SF_OPTIONS, "sfSchema": os.environ["SF_SCHEMA_ANALYTICS"]}

java_import(spark._jvm, "net.snowflake.spark.snowflake.Utils")
SfUtils = spark._jvm.net.snowflake.spark.snowflake.Utils

def query_obt(sql):
    return spark.read.format("snowflake").options(**SF_OPTS_ANALYTICS).option("query", sql).load()

def run_dml(sql):
    SfUtils.runQuery(SF_OPTS_ANALYTICS, sql)

print("✓ Configuración lista")

CELDA 1: Inicialización Spark + Configuración Snowflake
✓ SparkSession creada
✓ Configuración lista


## FASE 1: Análisis Exploratorio de Datos (EDA)
Exploraremos el estado actual de los datos sin modificarlos. Revisaremos proporciones de nulos, conteo de errores lógicos y calcularemos los límites estadísticos para outliers.

In [11]:
print("=" * 60)
print("CELDA 2: EDA - Nulos e Inconsistencias Lógicas")
print("=" * 60)

eda_sql = """
SELECT 
    COUNT(*) AS total_rows,
    
    -- Nulos
    ROUND(SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS pct_null_passenger,
    ROUND(SUM(CASE WHEN pickup_datetime IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS pct_null_pickup,
    ROUND(SUM(CASE WHEN pu_location_id IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS pct_null_puloc,
    
    -- Errores lógicos
    SUM(CASE WHEN dropoff_datetime < pickup_datetime THEN 1 ELSE 0 END) AS inverted_dates,
    SUM(CASE WHEN trip_duration_min < 0 THEN 1 ELSE 0 END) AS negative_duration,
    SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS negative_distance,
    SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) AS negative_amount
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
"""
print(">>> Resultados del EDA General (Nulos y Errores):")
df_eda = query_obt(eda_sql)
df_eda.show(truncate=False)

eda_results = df_eda.collect()[0]


CELDA 2: EDA - Nulos e Inconsistencias Lógicas
>>> Resultados del EDA General (Nulos y Errores):
+----------+------------------+---------------+--------------+--------------+-----------------+-----------------+---------------+
|TOTAL_ROWS|PCT_NULL_PASSENGER|PCT_NULL_PICKUP|PCT_NULL_PULOC|INVERTED_DATES|NEGATIVE_DURATION|NEGATIVE_DISTANCE|NEGATIVE_AMOUNT|
+----------+------------------+---------------+--------------+--------------+-----------------+-----------------+---------------+
|815553850 |0.00              |0.00           |0.00          |87701         |13956            |30967            |2972628        |
+----------+------------------+---------------+--------------+--------------+-----------------+-----------------+---------------+



In [12]:
print("=" * 60)
print("CELDA 3: EDA - Detección Estadística de Outliers (IQR)")
print("=" * 60)

iqr_sql = """
SELECT 
    APPROX_PERCENTILE(trip_duration_min, 0.25) AS dur_q1,
    APPROX_PERCENTILE(trip_duration_min, 0.75) AS dur_q3,
    APPROX_PERCENTILE(trip_distance, 0.25) AS dist_q1,
    APPROX_PERCENTILE(trip_distance, 0.75) AS dist_q3,
    APPROX_PERCENTILE(total_amount, 0.25) AS amt_q1,
    APPROX_PERCENTILE(total_amount, 0.75) AS amt_q3
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
"""
print(">>> Computando estadísticos aproximados para definir umbrales de outliers...")
df_iqr = query_obt(iqr_sql).collect()[0]

dur_upper = df_iqr['DUR_Q3'] + (3 * (df_iqr['DUR_Q3'] - df_iqr['DUR_Q1']))
dist_upper = df_iqr['DIST_Q3'] + (3 * (df_iqr['DIST_Q3'] - df_iqr['DIST_Q1']))
amt_upper = df_iqr['AMT_Q3'] + (3 * (df_iqr['AMT_Q3'] - df_iqr['AMT_Q1']))

print("Umbrales empíricos reportados (Q3 + 3*IQR):")
print(f"- Límite Outlier Duración (min): {dur_upper}")
print(f"- Límite Outlier Distancia (mi): {dist_upper}")
print(f"- Límite Outlier Monto Total ($): {amt_upper}")


CELDA 3: EDA - Detección Estadística de Outliers (IQR)
>>> Computando estadísticos aproximados para definir umbrales de outliers...
Umbrales empíricos reportados (Q3 + 3*IQR):
- Límite Outlier Duración (min): 55.00000000000004
- Límite Outlier Distancia (mi): 9.952643477523212
- Límite Outlier Monto Total ($): 55.5576936730792


## FASE 2: Estrategia de Limpieza Dinámica
Con base en los resultados del EDA, el código estructurará y ejecutará dinámicamente las sentencias de limpieza:
1. Si existen nulos en `passenger_count`, se imputarán con la mediana típica (1).
2. Los nulos en variables de tiempo o geografía se eliminan (Listwise deletion).
3. Se descartan errores lógicos.
4. Se aplican los límites calculados de `3*IQR` para podar la larga cola de outliers.

In [ ]:
print("=" * 60)
print("CELDA 4: Ejecución Dinámica de Limpieza")
print("=" * 60)

print(">>> Analizando resultados del EDA para armar plan de limpieza...")
queries_to_run = []

# 1. Estrategia para Nulos en passenger_count (Imputación si hay nulos)
if eda_results['PCT_NULL_PASSENGER'] > 0:
    print(f"-> Detectado {eda_results['PCT_NULL_PASSENGER']}% nulos en passenger_count. Estrategia: Imputación con mediana (1).")
    queries_to_run.append("""
        UPDATE NYC_TAXI_P3.ANALYTICS.OBT_TRIPS 
        SET passenger_count = 1 
        WHERE passenger_count IS NULL OR passenger_count = 0
    """)

# 2. Estrategia para Nulos Esenciales (Eliminación)
if eda_results['PCT_NULL_PICKUP'] > 0 or eda_results['PCT_NULL_PULOC'] > 0:
    print("-> Detectados nulos en variables clave. Estrategia: Listwise deletion.")
    queries_to_run.append("""
        DELETE FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS 
        WHERE pickup_datetime IS NULL 
           OR dropoff_datetime IS NULL 
           OR pu_location_id IS NULL 
           OR do_location_id IS NULL
    """)

# 3. Estrategia para Errores Lógicos
if (eda_results['INVERTED_DATES'] > 0 or eda_results['NEGATIVE_DURATION'] > 0 or 
    eda_results['NEGATIVE_DISTANCE'] > 0 or eda_results['NEGATIVE_AMOUNT'] > 0):
    print("-> Detectados errores lógicos (fechas invertidas, valores negativos). Estrategia: Filtrado duro.")
    queries_to_run.append("""
        DELETE FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
        WHERE dropoff_datetime < pickup_datetime
           OR total_amount < 0
           OR fare_amount < 0
           OR trip_distance < 0
           OR trip_duration_min < 0
           OR EXTRACT(YEAR FROM pickup_datetime) < 2010
           OR EXTRACT(YEAR FROM pickup_datetime) > 2026
    """)

# 4. Estrategia para Outliers Estadísticos
print(f"-> Aplicando umbrales de outliers IQR para distancia, duración y monto...")
queries_to_run.append(f"""
    DELETE FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
    WHERE trip_duration_min > {dur_upper}
       OR trip_distance > {dist_upper}
       OR total_amount > {amt_upper}
""")

print("\n>>> Ejecutando queries de limpieza en Snowflake...")
for i, sql in enumerate(queries_to_run):
    print(f"Ejecutando paso {i+1}...")
    run_dml(sql)

print("✓ Limpieza Dinámica Completada")


## FASE 3: Resumen Final Post-Limpieza
Verificación final de la tabla para confirmar que se aplicaron los criterios correctos.

In [ ]:
print("=" * 60)
print("CELDA 5: Resumen de Calidad Final")
print("=" * 60)

quality_sql = """
SELECT
    COUNT(*)                                             AS total_rows,
    ROUND(AVG(trip_duration_min), 2)                     AS avg_duration_min,
    ROUND(AVG(trip_distance), 2)                         AS avg_distance_mi,
    ROUND(AVG(total_amount), 2)                          AS avg_total_amount,
    MAX(trip_duration_min)                               AS max_duration,
    MAX(trip_distance)                                   AS max_distance,
    MAX(total_amount)                                    AS max_amount,
    MIN(trip_duration_min)                               AS min_duration,
    MIN(total_amount)                                    AS min_amount
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
"""
query_obt(quality_sql).show(truncate=False)
print("✓ Proceso completo de EDA y Limpieza")